# Topic 2: Sources & Sinks

> **Phase 6 → Week 10 → Topic 2**

---

## Built-in Sources

```
Source          Format          Purpose
──────────────  ──────────────  ──────────────────────────────────────────────
rate            built-in        Testing: generates (timestamp, value) at N rows/sec
rate-micro-batch built-in       Like rate but generates per micro-batch (Spark 3.3+)
file            JSON/CSV/       Watch a directory for new files
                Parquet/ORC
socket          TCP socket      Testing only: read text lines from nc -lk port
kafka           Kafka topics    Production: real-time event streams
delta           Delta table     Streaming from a Delta table (CDF required)
```

## Built-in Sinks

```
Sink            Exactly-once?   Purpose
──────────────  ──────────────  ──────────────────────────────────────────────
console         No              Development/debugging: prints to stdout
memory          No              Testing: queryable in-memory table
file            Yes             Production: write Parquet/Delta/CSV to disk/S3
kafka           At-least-once   Production: write events to Kafka topic
delta           Yes             Production: write to Delta table (idempotent)
foreach         Depends on impl Custom: write row-by-row to any system
foreachBatch    Yes if idempot. Custom: write a whole micro-batch as a DataFrame
```

---

## Interview Cheat Sheet

**Q: What is the file source and when do you use it?**
> The file source watches a directory for new files. When a new file appears, Spark reads it as the next micro-batch. Use it for: log files dropped by an upstream system, S3 prefixes where files land on a schedule, or any file-drop integration pattern. Spark tracks which files were already processed via the checkpoint — it will never reprocess a file.

**Q: What is the difference between `foreach` and `foreachBatch`?**
> `foreach` calls a user function once per **row** — good for writing to REST APIs or custom row-level sinks. `foreachBatch` calls a user function once per **micro-batch**, passing the entire batch as a static DataFrame — this is far more efficient because you can use batch DataFrame APIs (JDBC bulk insert, Delta MERGE, etc.) and the batch runs with full Catalyst optimization.

**Q: Which sinks guarantee exactly-once delivery?**
> File sink and Delta sink guarantee exactly-once because they write atomically and use the checkpoint to ensure each batch is committed at most once. Kafka sink is at-least-once. `foreachBatch` is exactly-once IF your batch write operation is idempotent.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time, os, shutil, json

spark = SparkSession.builder \
    .appName("Week10 - Sources and Sinks") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

## Part 1: File Source — Watch a Directory

In [ ]:
FILE_SRC = "/tmp/stream_json_in"
FILE_OUT = "/tmp/stream_json_out"
CKPT    = "/tmp/ckpt_file_source"

for p in [FILE_SRC, FILE_OUT, CKPT]:
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p)

# Drop batch 0 before starting the stream
def write_batch(n, prefix):
    rows = [(f"{prefix}-{i:03d}", float(i * n)) for i in range(1, 6)]
    spark.createDataFrame(rows, ["event_id", "amount"]) \
         .coalesce(1).write.json(f"{FILE_SRC}/batch_{n}")
    print(f"Wrote batch {n}")

write_batch(1, "A")
write_batch(2, "B")

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("amount", DoubleType()),
])

file_stream = spark.readStream \
    .schema(event_schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(FILE_SRC)

q = file_stream \
    .withColumn("tax", F.col("amount") * 0.18) \
    .writeStream \
    .format("memory") \
    .queryName("file_results") \
    .outputMode("append") \
    .option("checkpointLocation", CKPT) \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(6)
print("After batch 1 and 2:")
spark.sql("SELECT * FROM file_results ORDER BY event_id").show()

# Drop batch 3 while query is running — Spark auto-picks it up
write_batch(3, "C")
time.sleep(4)
print("After batch 3 (added while running):")
spark.sql("SELECT count(*) as total FROM file_results").show()

q.stop()

## Part 2: Console Sink — Development/Debugging

In [ ]:
# Console sink: prints each micro-batch to stdout
# Only for development — not for production
stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 3) \
    .load() \
    .withColumn("squared", F.col("value") ** 2)

q_console = stream.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("numRows", 5) \
    .option("truncate", False) \
    .trigger(processingTime="2 seconds") \
    .start()

time.sleep(7)
q_console.stop()
print("Console output appears above — each batch prefixed with 'Batch: N'")

## Part 3: foreachBatch — The Most Powerful Sink

In [ ]:
# foreachBatch: called with (micro_batch_df, batch_id) every trigger
# The batch_df is a STATIC DataFrame — use all batch APIs

batch_log = []  # collect batch stats

def process_batch(batch_df, batch_id):
    count = batch_df.count()
    total = batch_df.agg(F.sum("value")).collect()[0][0] or 0
    batch_log.append({"batch_id": batch_id, "rows": count, "sum": total})
    print(f"  Batch {batch_id}: {count} rows, sum={total:.0f}")
    # In production: batch_df.write.format("delta").mode("append").save(path)
    # Or: batch_df.write.jdbc(...) for JDBC sinks
    # Or: call an external API for each partition

stream2 = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

q_foreach = stream2.writeStream \
    .foreachBatch(process_batch) \
    .trigger(processingTime="2 seconds") \
    .option("checkpointLocation", "/tmp/ckpt_foreach") \
    .start()

time.sleep(8)
q_foreach.stop()

print("\nBatch log:")
for b in batch_log:
    print(f"  batch {b['batch_id']}: {b['rows']} rows, sum={b['sum']:.0f}")

## Part 4: File Sink with Parquet — Production Pattern

In [ ]:
PARQUET_OUT = "/tmp/stream_parquet_out"
CKPT2 = "/tmp/ckpt_parquet"
for p in [PARQUET_OUT, CKPT2]:
    if os.path.exists(p): shutil.rmtree(p)

stream3 = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 20) \
    .load() \
    .withColumn("event_date", F.to_date(F.col("timestamp"))) \
    .withColumn("hour", F.hour(F.col("timestamp")))

q_parquet = stream3.writeStream \
    .format("parquet") \
    .option("path", PARQUET_OUT) \
    .option("checkpointLocation", CKPT2) \
    .outputMode("append") \
    .partitionBy("event_date") \
    .trigger(processingTime="3 seconds") \
    .start()

time.sleep(10)
q_parquet.stop()

print("Files written:")
for root, dirs, files in os.walk(PARQUET_OUT):
    for f in files:
        if not f.startswith("."):
            print("  ", os.path.join(root, f).replace(PARQUET_OUT, ""))

print("\nReading back:")
spark.read.parquet(PARQUET_OUT).count()

## Part 5: Source & Sink Quick Reference

In [ ]:
print("""
Source Options Reference:
════════════════════════════════════════════════════════════════

Rate source (testing):
  spark.readStream.format("rate")
    .option("rowsPerSecond", 100)   # rows generated per second
    .option("numPartitions", 4)     # parallelism
    .load()

File source (JSON/CSV/Parquet):
  spark.readStream
    .schema(my_schema)              # REQUIRED for streaming
    .option("maxFilesPerTrigger", 10)  # process at most 10 files per batch
    .option("latestFirst", True)    # process newest files first
    .json("/path/to/dir/")

Kafka source (covered in Week 11):
  spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "my-topic")
    .option("startingOffsets", "earliest")
    .load()

════════════════════════════════════════════════════════════════

Sink Options Reference:
════════════════════════════════════════════════════════════════

Console (dev only):         .format("console")
Memory (testing):           .format("memory").queryName("table_name")
Parquet/Delta (production): .format("parquet") / .format("delta")
                             .option("path", "/output/path")
foreachBatch (custom):      .foreachBatch(my_fn)

Always set checkpointLocation:
  .option("checkpointLocation", "/durable/path/checkpoint")
════════════════════════════════════════════════════════════════
""")

## Exercises

1. Create a file source that reads JSON files from a directory. Use `maxFilesPerTrigger=1`. Write 5 files before starting the query. Start the query and watch how it processes them one at a time. Then add 2 more files and verify Spark picks them up automatically.
2. Use `foreachBatch` to write each micro-batch to two different locations — a Parquet archive AND a CSV summary. (Hint: call `.cache()` on `batch_df` before writing twice, then `.unpersist()`)
3. Explain why the `memory` sink is not suitable for production. What would happen if the process restarts?
4. What is the `maxFilesPerTrigger` option and why would you set it? What happens if you omit it and 10,000 files land at once?
5. You need to write streaming data to a JDBC database. Which sink pattern do you use and why is it better than `foreach`?